# Sensitivity functions — S, T, PS, CS, and the loop under disturbance and noise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/topics/classical_control/sensitivity_functions.ipynb)

A feedback loop does more than track a reference. It also decides what a **load disturbance** $w$ does to the output and what **measurement noise** $v$ does to the actuator. Four transfer functions tell the whole story, read with the [minilink](https://github.com/alx87grd/minilink) API:

- the plant $H(s)$, input torque to measured angle;
- the controller $C(s)$, error to torque;
- an optional filter $F(s)$ on the measurement ($F = 1$ here);
- the loop gain $L(s) = C(s)\,H(s)\,F(s)$.

| function | formula | from → to | minilink |
| --- | --- | --- | --- |
| sensitivity | $S = \dfrac{1}{1 + L}$ | output disturbance → $y$, and $r \to e$ | `sensitivity` |
| complementary sensitivity | $T = \dfrac{C H}{1 + L}$ | $r \to y$, and $v \to y$ (sign aside) | `complementary_sensitivity` |
| load sensitivity | $PS = \dfrac{H}{1 + L}$ | $w \to y$ | `load_sensitivity` |
| noise sensitivity | $CS = \dfrac{C F}{1 + L}$ | $v \to u$ | `noise_sensitivity` |

Each function takes the pieces by name, `sensitivity(plant=H, controller=C)`, and returns a `TransferFunction`, so `bode` and `plot_bode` read it like any other system. Then `closed_loop(C, plant, w=True, v=True)` builds the same loop as a diagram with two extra inputs, and sine sources wired into them check the predictions in simulation.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import PID, DiagramSystem, Pendulum, Sine, closed_loop
from minilink.analysis import (
    bode,
    complementary_sensitivity,
    frequency_response,
    load_sensitivity,
    noise_sensitivity,
    plot_bode,
    sensitivity,
)

BACKEND = "matplotlib"  # or "plotly" (hover; the one to use in Colab)
# BACKEND = "plotly"

LOAD_AMPLITUDE, LOAD_OMEGA = 1.0, 1.0  # load torque w: N·m, rad/s
NOISE_AMPLITUDE, NOISE_OMEGA = 0.05, 50.0  # angle noise v: rad, rad/s

## 1. The pieces of the loop

The plant is the pendulum hanging down, $(m l^2 + I)\,\ddot\theta = \tau - m g l \sin\theta - d\,\dot\theta$, linearized about $\bar x = 0$. Its output `y` is the whole state $(\theta, \dot\theta)$, so we pick the angle channel `q`: $H(s)$ is torque to angle. The controller is a PID with a filtered derivative, $C(s) = K_p + \dfrac{K_i}{s} + \dfrac{K_d\, s}{\tau s + 1}$, on the error $e = r - \theta$.

In [ ]:
plant = Pendulum()
plant.params["d"] = 0.5  # a little damping
x_bar = np.zeros(2)  # hanging down, at rest
plant.x0 = x_bar

H = plant.transfer_function(x_bar, of="q")  # torque -> angle
C = PID(Kp=20.0, Ki=10.0, Kd=2.0, tau=0.05)  # error -> torque

The four functions, from the pieces. The body of each one writes $L = C H F$ and then the formula of the table above.

In [ ]:
S = sensitivity(plant=H, controller=C)
T = complementary_sensitivity(plant=H, controller=C)
PS = load_sensitivity(plant=H, controller=C)
CS = noise_sensitivity(plant=H, controller=C)

## 2. Reading the four Bode diagrams

**Sensitivity $S$.** Below the crossover, $|L| \gg 1$ and $|S| \approx 1/|L|$ is small: the integral action makes $S(0) = 0$, so a constant disturbance on the output is fully rejected. Above the crossover, $|L| \ll 1$ and $|S| \to 1$: feedback no longer acts. The notch at 2.2 rad/s is the pendulum's own resonance, where $|L|$ is largest. Near 4 rad/s, $|S|$ peaks at about +10 dB. That peak, $M_s = \max_\omega |S(j\omega)|$, is the inverse of the distance from $L(j\omega)$ to the critical point $-1$: here about 3, so $L$ passes within 0.3 of $-1$, and this loop is lightly damped. A smaller peak means a more robust loop.

In [ ]:
plot_bode(S, margins=False, backend=BACKEND, title="S(s): output disturbance -> y")

**Complementary sensitivity $T$.** The mirror image of $S$: $|T| \approx 1$ at low frequency (the angle follows the reference), the same resonant peak near 4 rad/s (an overshoot in the step response), and a roll-off above it, where measurement noise is filtered out of the angle.

In [ ]:
plot_bode(T, margins=False, backend=BACKEND, title="T(s): r -> y")

With $F = 1$ the two add up to one at every frequency, $S + T = \dfrac{1 + L}{1 + L} = 1$. This is the basic trade-off of feedback: $S$ and $T$ cannot both be small at the same frequency.

In [ ]:
w = np.logspace(-2, 3, 200)
_, S_jw = frequency_response(S, w=w)
_, T_jw = frequency_response(T, w=w)
np.allclose(S_jw + T_jw, 1.0)

**Load sensitivity $PS$.** A torque disturbance $w$ enters at the plant input. At low frequency the integral action rejects it; at high frequency the plant's own inertia filters it. It is worst near the closed-loop resonance, around 4 rad/s, the frequency where $|S|$ peaks too.

In [ ]:
plot_bode(PS, margins=False, backend=BACKEND, title="PS(s): load torque w -> y")

**Noise sensitivity $CS$.** Measurement noise $v$ reaches the torque command through the controller. At high frequency $|CS| \to |C(j\omega)|$, and the derivative term makes $|C|$ large there: $K_p + K_d/\tau = 60$ N·m/rad, about +36 dB. The angle barely moves ($T$ is small up there), but the actuator chatters.

In [ ]:
plot_bode(CS, margins=False, backend=BACKEND, title="CS(s): angle noise v -> u")

## 3. A specification is one `bode` call

Specifications are written as bounds on these functions at given frequencies. `bode` takes the frequency in rad/s; a frequency $f$ in hertz is $\omega = 2\pi f$.

*The load torque at 1 rad/s reaches the angle attenuated below −25 dB:*

In [ ]:
_, PS_db, _ = bode(PS, w=LOAD_OMEGA)
PS_db[0] < -25.0

*Angle noise at 50 rad/s reaches the torque amplified by at most 20 dB:* this one fails, because of the derivative gain. A longer derivative filter $\tau$, or a filter $F$ on the measurement (`filter=F`), trades a little phase margin for a quieter actuator.

In [ ]:
_, CS_db, _ = bode(CS, w=NOISE_OMEGA)
CS_db[0] < 20.0

## 4. The same loop, simulated with a disturbance and noise

`closed_loop(C, plant, ...)` builds the loop that `C @ plant` builds, with optional boundary inputs: `r=True` (the reference, on by default), `w=True` (a torque summed into the plant input) and `v=True` (noise summed into the measured angle). Here the reference stays at zero (`r=False`): we look at regulation only. Two extra outputs expose the angle and the torque command.

In [ ]:
loop = closed_loop(C, plant, r=False, w=True, v=True)
loop.connect_new_output_port("sys", "q", "theta")
loop.connect_new_output_port("ctl", "u", "tau")

In [ ]:
loop.plot_diagram()

Two `Sine` sources, $w(t) = 1.0 \sin(1.0\, t)$ N·m and $v(t) = 0.05 \sin(50\, t)$ rad, drive the loop's `w` and `v` ports in a diagram.

In [ ]:
diagram = DiagramSystem()
diagram.add_subsystem(Sine(amplitude=LOAD_AMPLITUDE, omega=LOAD_OMEGA), "load")
diagram.add_subsystem(Sine(amplitude=NOISE_AMPLITUDE, omega=NOISE_OMEGA), "sensor_noise")
diagram.add_subsystem(loop, "loop")
diagram.connect("load", "y", "loop", "w")
diagram.connect("sensor_noise", "y", "loop", "v")

In [ ]:
diagram.plot_diagram()

In [ ]:
traj = diagram.compute_trajectory(tf=30.0, dt=0.001, verbose=False)

In [ ]:
diagram.plot_trajectory(
    traj, signals=("load:y", "sensor_noise:y", "loop:theta", "loop:tau")
)

## 5. The prediction, checked

By superposition, in steady state the angle is the load sine scaled by $|PS(j\omega_w)|$, plus the noise sine scaled by $|T(j\omega_v)|$. The torque carries the noise scaled by $|CS(j\omega_v)|$. The linear prediction of those two amplitudes:

In [ ]:
theta_from_load = LOAD_AMPLITUDE * 10 ** (PS_db[0] / 20)  # rad
tau_from_noise = NOISE_AMPLITUDE * 10 ** (CS_db[0] / 20)  # N·m
theta_from_load, tau_from_noise

The simulation of the **nonlinear** pendulum agrees: after the transient, the angle swings by about ±0.035 rad at 1 rad/s (the noise adds only $0.05 \cdot |T(j50)| \approx 0.0006$ rad), and the torque carries a ±2.8 N·m chatter at 50 rad/s on top of the slow load compensation. Small angles keep the pendulum in its linear regime, so the Bode prediction holds. Raise `LOAD_AMPLITUDE` until the angle leaves that regime, and watch the prediction drift.

## 6. Recap

| question | minilink | the math |
| --- | --- | --- |
| what an output disturbance does to $y$ | `sensitivity(plant=H, controller=C)` | $S = 1/(1 + L)$ |
| how $y$ follows $r$ | `complementary_sensitivity(...)` | $T = CH/(1 + L)$, $S + T = 1$ |
| what a load torque does to $y$ | `load_sensitivity(...)` | $PS = H/(1 + L)$ |
| what sensor noise does to $u$ | `noise_sensitivity(...)` | $CS = CF/(1 + L)$ |
| a specification at one frequency | `bode(G, w=2 * np.pi * f)` | $20 \log_{10} \lvert G(j\omega) \rvert$ |
| the loop with disturbance and noise inputs | `closed_loop(C, plant, w=True, v=True)` | $u_{plant} = u + w$, $y_{meas} = y + v$ |
| a sine into a port | `Sine(amplitude, omega)` + `diagram.connect` | $a \sin(\omega t + \varphi)$ |